In [ ]:
%load_ext autoreload
%autoreload 2
import baostock as bs
import pandas as pd
import os
import numpy as np
from scipy.stats import rankdata


In [ ]:
#### 登陆系统 ####
def accquire_stock(code='sh.600000',data_type='d',start_date='2024-07-01',end_date='2024-12-31'):
    lg = bs.login()
    # 显示登陆返回信息
    print('login respond error_code:'+lg.error_code)
    print('login respond  error_msg:'+lg.error_msg)

    #### 获取沪深A股历史K线数据 ####
    # 详细指标参数，参见"历史行情指标参数"章节；"分钟线"参数与"日线"参数不同。"分钟线"不包含指数。
    # 分钟线指标：date,time,code,open,high,low,close,volume,amount,adjustflag
    # 周月线指标：date,code,open,high,low,close,volume,amount,adjustflag,turn,pctChg
    #日线指标：date,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
    data_aquire_list=({
                          'd':"date,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST",
                          'm':'date,code,open,high,low,close,volume,amount,adjustflag,turn,pctChg',
                          '5':'date,time,code,open,high,low,close,volume,amount,adjustflag'
                      }.get(data_type))
    rs = bs.query_history_k_data_plus(code,
        data_aquire_list,
        start_date, end_date,
        frequency=data_type, adjustflag="3")
    print('query_history_k_data_plus respond error_code:'+rs.error_code)
    print('query_history_k_data_plus respond  error_msg:'+rs.error_msg)

    #### 打印结果集 ####
    data_list = []
    while (rs.error_code == '0') & rs.next():
        # 获取一条记录，将记录合并在一起
        data_list.append(rs.get_row_data())
    result = pd.DataFrame(data_list, columns=rs.fields)

    #### 登出系统 ####
    bs.logout()
    return result




In [ ]:
import numpy as np

#清洗数据模块    data_type代表获取的是日线还是分钟线(D,M)不区分大小写
def data_clean(df,data_type):
    data_type=data_type.upper()
    data_list_group_by_type={'D':{'price_cols': ['open', 'high', 'low', 'close','preclose'],
                                  'volume_cols':['volume', 'amount','turn'],
                                  'unic_cols':['pctChg','tradestatus']
                                  },
                             'M':{'price_cols':['open', 'high', 'low', 'close'], #关于日和分钟线的交易量相关数据的表头与价格相关数据的表头
                                  'volume_cols':['volume', 'amount']
                                  }
                             }
    price_cols = data_list_group_by_type[data_type]['price_cols']
    volume_cols = data_list_group_by_type[data_type]['volume_cols']
    unic_cols=data_list_group_by_type[data_type].get('unic_cols',[])
    for col in price_cols + volume_cols + unic_cols + ['adjustflag']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    #转化为时间序列
    #分钟线：df['time'] = pd.to_datetime(df['time'],format='%Y%m%d%H%M%S%f')
    #日线：df['date'] = pd.to_datetime(df['date'])
    if data_type =='D':
        df['date'] = pd.to_datetime(df['date'])
    elif data_type =='M':
        df['time'] = pd.to_datetime(df['time'],format='%Y%m%d%H%M%S%f')
    #检测是否有数据缺失，若有缺失则向前一个数据填充
    df['code'] = df['code'].ffill()
    df['open'] = df.groupby('code')['open'].ffill()
    df['high'] = df.groupby('code')['high'].ffill()
    df['low'] = df.groupby('code')['low'].ffill()
    df['close'] = df.groupby('code')['close'].ffill()
    df['preclose'] = df.groupby('code')['preclose'].ffill()
    #价格指标负数处理，若为负数则标记为nan
    for col in price_cols:
        df.loc[df[col] <= 0, col] = np.nan
    #规定停牌时的换手率和涨跌幅为0
    mask_stop = (df['tradestatus'] == 0)
    df.loc[mask_stop, 'turn'] = 0.0
    df.loc[mask_stop, 'pctChg'] = 0.0


    #  删除所有价格或成交量全为空的行
    df.dropna(subset=['close', 'volume'], how='all', inplace=True)
    #若high<low，则两者对调
    df['high'], df['low'] = np.where(df['high'] < df['low'],
                                 [df['low'], df['high']],
                                 [df['high'], df['low']])
    #保证low<=open,close<=high,超出则对应用low或high代替
    df['open'] = np.where(df['open'] > df['high'], df['high'], df['open'])
    df['open'] = np.where(df['open'] < df['low'], df['low'], df['open'])
    df['close'] = np.where(df['close'] > df['high'], df['high'], df['close'])
    df['close'] = np.where(df['close'] < df['low'], df['low'], df['close'])

    #更改所有无成交量 负成交量 无成交额 负成交额 为0
    df['volume'] = df['volume'].fillna(0).clip(lower=0)
    df['amount'] = df['amount'].fillna(0).clip(lower=0)

#异常量检测

    #通过 成交量*收盘价 估算 成交额 计算 与成交额的差距百分比 ，并标记误差超过10%的（day）
    df['amount_est'] = df['volume'] * df['close']
    df['amount_error_ratio'] = abs(df['amount'] - df['amount_est']) / df['amount_est'].replace(0, np.nan)
    df['amount_suspicious'] = df['amount_error_ratio'] > 0.10

    #涨跌幅缺失且正常交易时，计算涨跌幅
    df.loc[df['pctChg'].isna() & (df['tradestatus'] == 1), 'pctChg'] = df.loc[df['pctChg'].isna() & (df['tradestatus'] == 1), 'close'] / df.loc[df['pctChg'].isna() & (df['tradestatus'] == 1), 'preclose'] -1

    #换手率通常在0到0.8之间，超过的数据截断
    df.loc[(df['tradestatus'] == 1) & (df['turn'] > 0.8), 'turn'] = 0.8
    df.loc[(df['tradestatus'] == 1) & (df['turn'] < 0), 'turn'] = 0.0

    #删除同一时间的同股票代码数据，仅保留最后一项
    #分钟线：df.drop_duplicates(subset=['time', 'code'], keep='last', inplace=True)
    #日线：df.drop_duplicates(subset=['date', 'code'], keep='last', inplace=True)
    if data_type == 'D':
        df.drop_duplicates(subset=['date', 'code'], keep='last', inplace=True)
    elif data_type =='M':
        df.drop_duplicates(subset=['time', 'code'], keep='last', inplace=True)
    return df

#获取对应baoshock里的对应股票代码的数据，清洗，存储为csv文件
def result_to_csv(result,data_type):
    result=data_clean(result,data_type=data_type)
    fname = "{}_{}_to_{}_{}.csv".format(
        result.loc[0,'code'],
        result.loc[0,'date'].date(),
        result.loc[result.index[-1],'date'].date(),
        data_type)
    result.to_csv(fname, index=False)
    return result

In [ ]:
#五只银行股
result_to_csv(accquire_stock('sh.600000','d','2024-07-01','2024-12-31')
                            ,'d').copy()

result_to_csv(accquire_stock('sh.601988','d','2024-07-01','2024-12-31')
                            ,'d').copy()

result_to_csv(accquire_stock('sh.601939','d','2024-07-01','2024-12-31')
                            ,'d').copy()

result_to_csv(accquire_stock('sh.601928','d','2024-07-01','2024-12-31')
                            ,'d').copy()

result_to_csv(accquire_stock('sh.601288','d','2024-07-01','2024-12-31')
                            ,'d').copy()


In [ ]:
#贵州茅台 中国平安 海康威视 恒瑞医药 京东方A
codes=['sh.600519','sh.601318','sz.002415','sh.600276','sz.000725']
for code in codes:
    result_to_csv(accquire_stock(code,'d','2024-07-01','2024-12-31')
                            ,'d')

In [ ]:
import pandas as pd
#读取csv文件，转化为时间序列
def csv_read(csv_path):
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    return df.copy()
#五只银行股
result_601288=csv_read('sh.601288_2024-07-01_to_2024-12-31_d.csv')
result_600000=csv_read('sh.600000_2024-07-01_to_2024-12-31_d.csv')
result_601928=csv_read('sh.601928_2024-07-01_to_2024-12-31_d.csv')
result_601939=csv_read('sh.601939_2024-07-01_to_2024-12-31_d.csv')
result_601988=csv_read('sh.601988_2024-07-01_to_2024-12-31_d.csv')


result_600519=csv_read('sh.600519_2024-07-01_to_2024-12-31_d.csv')
result_601318=csv_read('sh.601318_2024-07-01_to_2024-12-31_d.csv')
result_002415=csv_read('sz.002415_2024-07-01_to_2024-12-31_d.csv')
result_600276=csv_read('sh.600276_2024-07-01_to_2024-12-31_d.csv')
result_000725=csv_read('sz.000725_2024-07-01_to_2024-12-31_d.csv')

In [ ]:
from alphas.alpha_001 import alpha_001
alpha_001_result=alpha_001(dfs=[result_600000,result_601288,result_601928,result_601939,result_601988],date_measure='2024-8-2')
print(alpha_001_result)

In [ ]:
from alphas.alpha_002 import alpha_002
alpha_002_result=alpha_002(dfs=[result_600000,result_601288,result_601928,result_601939,result_601988],date_measure='2024-7-11',correlation_day=6)
print(alpha_002_result)

In [ ]:
from alphas.alpha_003 import alpha_003
alpha_003_result=alpha_003(dfs=[result_600000,result_601288,result_601928,result_601939,result_601988],date_measure='2024-7-30',correlation_day=10)
print(alpha_003_result)

In [ ]:
from alphas.alpha_004 import alpha_004
alpha_004_result=alpha_004(dfs=[result_600000,result_601288,result_601928,result_601939,result_601988],date_measure='2024-7-17',ts_rank_day=9)
print(alpha_004_result)

In [ ]:
from alphas.alpha_005 import alpha_005
alpha_005_result=alpha_005(dfs=[result_600000,result_601288,result_601928,result_601939,result_601988],date_measure='2024-7-17',type='d')
print(alpha_005_result)

In [ ]:
from alphas.alpha_006 import alpha_006
alpha_006_result = alpha_006(dfs=[result_600000, result_601288, result_601928, result_601939, result_601988],
                            date_measure='2024-7-12', correlation_day=10)
print(alpha_006_result)

In [ ]:
from alphas.alpha_007 import alpha_007
alpha_007_result = alpha_007(dfs=[result_600000],
                            date_measure='2024-11-6')
print(alpha_007_result)

In [ ]:
from alphas.alpha_008 import alpha_008
alpha_008_result = alpha_008(dfs=[result_600000, result_601288, result_601928, result_601939, result_601988],
                            date_measure='2024-7-22')
print(alpha_008_result)

In [ ]:
from alphas.alpha_009 import alpha_009
alpha_009_result = alpha_009(dfs=[result_600000],
                            date_measure='2024-7-24')
print(alpha_009_result)

In [ ]:
from alphas.alpha_010 import alpha_010
alpha_010_result = alpha_010(dfs=[result_600000, result_601288, result_601928, result_601939, result_601988],
                            date_measure='2024-7-22')
print(alpha_010_result)

In [ ]:
# ===== IC 因子评估 =====
from utils.evaluation import (
    build_forward_return_panel,
    build_factor_panel,
    compute_rank_ic_series,
    ic_summary,
)

# --- 公共设置 ---
dfs = [result_600000, result_601288, result_601928, result_601939, result_601988,result_600519,result_601318,result_002415,result_600276,result_000725]

# 日期范围：给足历史数据余量，从 9 月开始
dates = pd.date_range('2024-09-01', '2024-12-27', freq='B')

# --- 未来收益面板（所有因子共用） ---
fwd_ret_df = build_forward_return_panel(dfs, dates)

# ==================== 遍历全部 10 个因子 ====================

# 因子列表：(显示名称, 模块路径, 函数名, 是否单股票, 额外参数)
alphas = [
    ("alpha_001", "alphas.alpha_001", "alpha_001", False, {}),
    ("alpha_002", "alphas.alpha_002", "alpha_002", False, {"correlation_day": 6}),
    ("alpha_003", "alphas.alpha_003", "alpha_003", False, {"correlation_day": 10}),
    ("alpha_004", "alphas.alpha_004", "alpha_004", False, {"ts_rank_day": 9}),
    ("alpha_005", "alphas.alpha_005", "alpha_005", False, {"type": "d"}),
    ("alpha_006", "alphas.alpha_006", "alpha_006", False, {"correlation_day": 10}),
    ("alpha_007", "alphas.alpha_007", "alpha_007", True,  {}),
    ("alpha_008", "alphas.alpha_008", "alpha_008", False, {}),
    ("alpha_009", "alphas.alpha_009", "alpha_009", True,  {}),
    ("alpha_010", "alphas.alpha_010", "alpha_010", False, {}),
]

for name, module_path, func_name, single, extra_kwargs in alphas:
    # 动态导入因子函数
    mod = __import__(module_path, fromlist=["dummy"])
    alpha_func = getattr(mod, func_name)

    # 构建因子面板
    f_df = build_factor_panel(alpha_func, dfs, dates,
                              single_stock=single, **extra_kwargs)

    # 计算 rank IC 序列
    ic = compute_rank_ic_series(f_df, fwd_ret_df)

    # 汇总统计
    s = ic_summary(ic)

    print(f"{name}:")
    print(f"  平均值: {s['mean_ic']:.6f}")
    print(f"  标准差: {s['std_ic']:.6f}")
    print(f"  IC_IR : {s['ic_ir']:.6f}")
    print(f"  有效样本: {ic.dropna().count()}")
    print()